# Notebook 23 — Progetto 2: generazione di immagini con Stable Diffusion

**Modulo:** 23 — Allineamento e generazione multimodale (**Progetto 2**)
**Tempo stimato:** 2-3 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Una pipeline `diffusers` funzionante per generare immagini da testo (text-to-image)
- Un esperimento guidato: prompt vago vs prompt dettagliato, e il numero di passi di
  inferenza (`num_inference_steps`) — gli stessi concetti visti in astratto nella dispensa
  (Sezioni 5.3 e 7.4), qui osservati sui pixel
- Un esercizio di inpainting sull'immagine di esempio già pronta in `assets/`, incluso il
  passaggio (spesso trascurato) di preparare la maschera nel formato corretto
- Una piccola pipeline image-to-image
- Una "galleria di confronto" riutilizzabile che misura tempo e qualità percepita al variare
  degli step
- Un **percorso alternativo via API** (senza GPU) per chi non ha una scheda video disponibile

## Prerequisiti

```bash
pip install diffusers transformers accelerate torch pillow numpy python-dotenv
# solo se vuoi anche il percorso alternativo via API:
pip install openai
```

Serve un file `.env` nella root del progetto con (a seconda del percorso che segui):
```
HUGGINGFACE_TOKEN=hf_...
OPENAI_API_KEY=sk-...
```

> **Attenzione — hardware:** `diffusers` con Stable Diffusion è pensato per **GPU (CUDA)**. Su
> CPU funziona, ma una singola immagine può richiedere diversi minuti invece di pochi secondi.
> Se non hai una GPU disponibile (es. Colab con runtime GPU attivo — `Runtime > Cambia tipo di
> runtime > GPU`), **salta direttamente alla sezione finale "Percorso alternativo: API senza
> GPU"**, che replica gli stessi concetti (text-to-image, prompt engineering, inpainting)
> usando `gpt-image-1` via API, sullo stesso pattern del Modulo 13.

---
## Cella 0 — Setup

Carichiamo le chiavi opzionali e rileviamo il device disponibile.

In [ ]:
import os
from dotenv import load_dotenv
import torch

load_dotenv()

HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN")
OPENAI_KEY = os.getenv("OPENAI_API_KEY")

def verifica_chiave(nome: str, valore: str | None) -> None:
    if valore:
        print(f"OK {nome}: {valore[:8]}...")
    else:
        print(f"NON IMPOSTATA {nome}")

verifica_chiave("HUGGINGFACE_TOKEN", HF_TOKEN)
verifica_chiave("OPENAI_API_KEY", OPENAI_KEY)

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"\nDevice disponibile: {DEVICE}")
if DEVICE == "cpu":
    print(
        "Nessuna GPU rilevata: la Sezione 1-5 (diffusers) sara' molto lenta. "
        "Considera di saltare al 'Percorso alternativo: API senza GPU' in fondo al notebook."
    )


---
## Sezione 1 — Carichiamo la pipeline di Stable Diffusion

`StableDiffusionPipeline` incapsula i quattro componenti descritti nella dispensa (Sezione 7):
VAE, text encoder, U-Net e scheduler. Non li gestiamo singolarmente — la pipeline li orchestra
per noi — ma sapere cosa fa ciascuno aiuta a interpretare gli effetti dei parametri che
useremo.

In [ ]:
from diffusers import StableDiffusionPipeline

# verifica l'ID modello aggiornato su huggingface.co: i repository vengono occasionalmente spostati
ID_MODELLO = "stable-diffusion-v1-5/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    ID_MODELLO,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    token=HF_TOKEN,
)
pipe = pipe.to(DEVICE)

print(f"Pipeline caricata su: {DEVICE}")


---
## Sezione 2 — Prima generazione da testo

Una funzione riutilizzabile, con un seed opzionale per rendere i confronti riproducibili
(fondamentale per l'esperimento guidato della prossima sezione: senza seed fisso, due immagini
diverse potrebbero derivare dal rumore di partenza, non dal parametro che stiamo confrontando).

In [ ]:
def genera_immagine_diffusione(
    prompt: str,
    nome_file: str,
    steps: int = 30,
    guidance_scale: float = 7.5,
    seed: int | None = None,
):
    """Genera un'immagine da un prompt testuale con Stable Diffusion e la salva su disco.
    guidance_scale controlla quanto la generazione aderisce al prompt (valori tipici 5-10):
    piu' alto = piu' fedele al testo, ma con meno varieta'."""
    generatore = None
    if seed is not None:
        generatore = torch.Generator(device=DEVICE).manual_seed(seed)

    risultato = pipe(
        prompt,
        num_inference_steps=steps,
        guidance_scale=guidance_scale,
        generator=generatore,
    )
    immagine = risultato.images[0]
    immagine.save(nome_file)
    print(f"Immagine salvata in: {nome_file}")
    return immagine


genera_immagine_diffusione(
    prompt="Un gatto arancione seduto su un davanzale, luce del mattino, stile fotorealistico",
    nome_file="gatto_diffusione.png",
    seed=42,
)


---
## Sezione 3 — Esperimento guidato: prompt vago vs prompt dettagliato

Lo stesso principio di prompt engineering visto per il testo (Moduli 02-03) e per le immagini
via API (Modulo 13) vale anche per un modello di diffusione che gira in locale: il livello di
dettaglio del prompt guida fortemente cosa il text encoder di CLIP "comunica" alla U-Net a ogni
passo (dispensa, Sezione 7.2).

**Prima di eseguire:** con lo stesso seed fisso, cosa ti aspetti che cambi tra le due immagini?
Solo lo stile, o anche l'inquadratura e la composizione?

In [ ]:
SEED_CONFRONTO = 123

PROMPT_VAGO = "Un robot"

PROMPT_DETTAGLIATO = (
    "Un piccolo robot domestico dal design arrotondato e colori pastello, "
    "fermo su un pavimento di legno in un soggiorno luminoso, luce naturale "
    "da una finestra laterale, dettagli metallici riflettenti, stile "
    "illustrazione digitale morbida."
)

genera_immagine_diffusione(PROMPT_VAGO, "robot_vago.png", seed=SEED_CONFRONTO)
genera_immagine_diffusione(PROMPT_DETTAGLIATO, "robot_dettagliato.png", seed=SEED_CONFRONTO)

print("\nApri i due file e confrontali: stesso seed, prompt diverso.")


**Le tue osservazioni** (modifica questa cella):

- Con il prompt vago, il modello ha scelto liberamente: ...
- Con il prompt dettagliato, cosa è cambiato oltre allo stile (inquadratura, ambientazione,
  luce)? ...
- A parità di seed, quanto "peso" ha avuto il prompt rispetto al rumore di partenza? ...

---
## Sezione 4 — Quanto contano i passi di inferenza

La dispensa (Sezione 5.3, "Regola pratica") spiega che rimuovere il rumore gradualmente, in
molti piccoli passi, produce risultati migliori che farlo in un solo passaggio. Verifichiamolo
generando la stessa immagine (stesso prompt, stesso seed) con un numero crescente di step, e
misurando anche il tempo impiegato.

In [ ]:
import time

PROMPT_TEST_STEPS = "Un faro sulla scogliera al tramonto, mare mosso, cielo arancione"
VALORI_STEPS = [5, 15, 30, 50]

for n_step in VALORI_STEPS:
    inizio = time.time()
    genera_immagine_diffusione(
        PROMPT_TEST_STEPS, f"faro_steps_{n_step}.png", steps=n_step, seed=7
    )
    durata = time.time() - inizio
    print(f"  steps={n_step:2d} -> {durata:.1f} secondi\n")

print(
    "Confronta i file faro_steps_5/15/30/50.png: nota dove il miglioramento visivo "
    "smette di valere l'aumento del tempo di calcolo (dispensa, Sez. 5.3 e 7.4)."
)


---
## Sezione 5 — Inpainting con la maschera fornita

Usiamo l'immagine e la maschera già preparate in `assets/23_mesa.png` e
`assets/23_mesa_mask.png` (un tavolo da pranzo in legno).

> **Attenzione al formato della maschera:** `diffusers` si aspetta una maschera **binaria**,
> dove il **bianco indica l'area da rigenerare** e il **nero l'area da preservare**. L'asset
> fornito, però, è la foto originale con una macchia nera dipinta sopra la zona da modificare —
> un formato "visivo", non binario. Prima di poterla usare dobbiamo convertirla: è un passaggio
> di preprocessing realistico, il tipo di dettaglio che capita spesso lavorando con asset non
> generati appositamente per una libreria specifica.

In [ ]:
import numpy as np
from PIL import Image

PERCORSO_IMMAGINE = "assets/23_mesa.png"
PERCORSO_OVERLAY = "assets/23_mesa_mask.png"


def crea_maschera_binaria_da_overlay(percorso_overlay: str, soglia: int = 15) -> Image.Image:
    """Converte un overlay (la foto originale con una macchia nera sopra la zona da
    modificare) in una maschera binaria in scala di grigi, nel formato atteso da diffusers:
    bianco (255) = area da rigenerare, nero (0) = area da preservare.
    Un pixel viene considerato parte della macchia se tutti e tre i canali RGB sono
    sotto la soglia (cioe' quasi puro nero) -- una foto reale difficilmente ha pixel
    cosi' scuri in modo uniforme, quindi la soglia isola bene la zona dipinta."""
    overlay = np.array(Image.open(percorso_overlay).convert("RGB"))
    e_zona_da_modificare = np.all(overlay < soglia, axis=-1)
    maschera_array = np.where(e_zona_da_modificare, 255, 0).astype(np.uint8)
    return Image.fromarray(maschera_array, mode="L")


immagine_originale = Image.open(PERCORSO_IMMAGINE).convert("RGB")
maschera_diffusers = crea_maschera_binaria_da_overlay(PERCORSO_OVERLAY)
maschera_diffusers.save("mesa_maschera_binaria.png")

print(f"Immagine: {immagine_originale.size}, maschera: {maschera_diffusers.size}")
print("Maschera binaria salvata in: mesa_maschera_binaria.png (apri il file per verificarla)")


In [ ]:
from diffusers import AutoPipelineForInpainting

pipe_inpaint = AutoPipelineForInpainting.from_pretrained(
    ID_MODELLO,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    token=HF_TOKEN,
).to(DEVICE)

# Regola della dispensa (Sezione 8.2, richiamata anche nel Modulo 13): il prompt deve descrivere
# l'immagine RISULTANTE INTERA, non solo l'oggetto da inserire nella zona trasparente/bianca.
PROMPT_INPAINTING = (
    "Un tavolo da pranzo in legno chiaro con sedie intorno, con al centro un grande "
    "vaso di ceramica bianca pieno di girasoli freschi, luce naturale, stile fotografico"
)

risultato_inpainting = pipe_inpaint(
    prompt=PROMPT_INPAINTING,
    image=immagine_originale,
    mask_image=maschera_diffusers,
    num_inference_steps=30,
    generator=torch.Generator(device=DEVICE).manual_seed(0),
).images[0]

risultato_inpainting.save("mesa_con_girasoli.png")
print("Risultato salvato in: mesa_con_girasoli.png")


**Esercizio — `# TODO`:** modifica `PROMPT_INPAINTING` per generare qualcos'altro al centro
del tavolo (una fruttiera, un vassoio di dolci, una lampada) e rilancia la cella precedente,
salvando il risultato con un nome file diverso.

In [ ]:
# TODO: scrivi un nuovo prompt che descriva l'immagine risultante intera (tavolo + nuovo oggetto)
mio_prompt_inpainting = ""  # es. "Un tavolo da pranzo in legno chiaro con al centro una fruttiera piena di frutta fresca"

# TODO: richiama pipe_inpaint con mio_prompt_inpainting, la stessa immagine_originale e maschera_diffusers,
# salva il risultato con un nome file diverso da "mesa_con_girasoli.png"


---
## Sezione 6 — Image-to-image: partire da un'immagine, non dal rumore puro

Come descritto nella dispensa (Sezione 8.1), invece di partire da rumore casuale puro possiamo
partire da un'immagine esistente e aggiungere solo una quantità controllata di rumore. Il
parametro `strength` regola quanto ci si allontana dall'originale: vicino a 0 mantiene
l'immagine quasi identica, vicino a 1 la trasforma quasi del tutto.

In [ ]:
from diffusers import AutoPipelineForImage2Image

pipe_img2img = AutoPipelineForImage2Image.from_pretrained(
    ID_MODELLO,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    token=HF_TOKEN,
).to(DEVICE)

for valore_strength in [0.3, 0.7]:
    risultato = pipe_img2img(
        prompt="Lo stesso tavolo da pranzo, ma in stile illustrazione ad acquerello",
        image=immagine_originale,
        strength=valore_strength,
        num_inference_steps=30,
        generator=torch.Generator(device=DEVICE).manual_seed(0),
    ).images[0]
    nome_file = f"mesa_acquerello_strength_{valore_strength}.png"
    risultato.save(nome_file)
    print(f"strength={valore_strength} -> {nome_file}")


---
## Sfida finale — Galleria di confronto riutilizzabile

Costruisci una funzione `genera_galleria_confronto` che automatizzi l'esperimento della
Sezione 4, e lo estenda: invece di variare solo `num_inference_steps`, deve permettere di
confrontare **qualunque combinazione** di parametri.

La funzione deve:

1. Ricevere un prompt, un seed fisso, e un dizionario di combinazioni da testare, ad esempio:
   ```python
   {"pochi_step_bassa_guidance": {"steps": 10, "guidance_scale": 5.0},
    "molti_step_alta_guidance": {"steps": 50, "guidance_scale": 12.0}}
   ```
2. Per ogni combinazione, generare l'immagine (stesso prompt e seed per tutte, così l'unica
   variabile è la combinazione di parametri), misurare il tempo impiegato con `time.time()`, e
   salvarla con un nome file che includa l'etichetta della combinazione.
3. Stampare una tabella riassuntiva con etichetta, parametri usati e tempo di generazione.
4. Ritornare un dizionario `{etichetta: {"percorso": ..., "secondi": ...}}`.

Completa i `# TODO`.

In [ ]:
def genera_galleria_confronto(
    prompt: str,
    combinazioni: dict[str, dict],
    seed: int = 0,
    prefisso_file: str = "galleria",
) -> dict:
    """Genera un'immagine per ogni combinazione di parametri e produce un report
    di confronto (tempo di generazione per combinazione)."""
    risultati = {}

    for etichetta, parametri in combinazioni.items():
        # TODO: estrai steps e guidance_scale da `parametri` (con .get e un default ragionevole)
        steps = parametri.get("steps", 30)
        guidance_scale = parametri.get("guidance_scale", 7.5)

        # TODO: misura il tempo con time.time() prima e dopo la chiamata a genera_immagine_diffusione
        inizio = time.time()
        nome_file = f"{prefisso_file}_{etichetta}.png"
        # genera_immagine_diffusione(prompt, nome_file, steps=steps, guidance_scale=guidance_scale, seed=seed)
        durata = time.time() - inizio

        risultati[etichetta] = {"percorso": nome_file, "secondi": round(durata, 1)}

    # TODO: stampa una tabella riassuntiva leggibile (etichetta, steps, guidance_scale, secondi)
    print(f"{'etichetta':30s} {'steps':>6s} {'guidance':>9s} {'secondi':>8s}")
    for etichetta, parametri in combinazioni.items():
        r = risultati[etichetta]
        print(
            f"{etichetta:30s} {parametri.get('steps', 30):6d} "
            f"{parametri.get('guidance_scale', 7.5):9.1f} {r['secondi']:8.1f}"
        )

    return risultati


# Prova la tua implementazione (decommenta la riga in genera_galleria_confronto per generare davvero)
# genera_galleria_confronto(
#     prompt="Un faro sulla scogliera al tramonto, mare mosso, cielo arancione",
#     combinazioni={
#         "pochi_step_bassa_guidance": {"steps": 10, "guidance_scale": 5.0},
#         "molti_step_alta_guidance": {"steps": 50, "guidance_scale": 12.0},
#     },
#     seed=7,
# )


---
## Percorso alternativo: API senza GPU

Se non hai una GPU disponibile, questa sezione replica gli stessi concetti — generazione da
testo, prompt engineering, inpainting con maschera — usando `gpt-image-1` via API, sullo stesso
pattern già visto nel Modulo 13 (che riusiamo qui, come richiede la coerenza del corso: non
reinventiamo `calcola_costo_immagine`, la riportiamo identica).

In [ ]:
from openai import OpenAI
import base64

client = OpenAI(api_key=OPENAI_KEY)


def genera_immagine_api(prompt: str, nome_file: str, size: str = "1024x1024", quality: str = "standard") -> str:
    """Genera un'immagine via API (vedi Modulo 13) -- l'equivalente, senza GPU locale,
    di genera_immagine_diffusione definita sopra."""
    risultato = client.images.generate(
        model="gpt-image-1",  # verifica l'ID modello aggiornato nella doc ufficiale
        prompt=prompt,
        size=size,
        quality=quality,
        n=1,
    )
    immagine_bytes = base64.b64decode(risultato.data[0].b64_json)
    with open(nome_file, "wb") as f:
        f.write(immagine_bytes)
    print(f"Immagine salvata in: {nome_file}")
    return nome_file


# Prezzi ILLUSTRATIVI in USD per immagine (stessa tabella del Modulo 13: verifica sempre i prezzi reali aggiornati)
PREZZI_IMMAGINE = {
    ("gpt-image-1", "1024x1024", "standard"): 0.04,
    ("gpt-image-1", "1024x1024", "high"):     0.08,
}


def calcola_costo_immagine(modello: str, size: str, quality: str, n_immagini: int = 1) -> dict:
    """Stessa funzione del Modulo 13: stima il costo di n immagini generate via API."""
    chiave = (modello, size, quality)
    if chiave not in PREZZI_IMMAGINE:
        raise ValueError(f"Combinazione non in tabella: {chiave}")
    prezzo_unitario = PREZZI_IMMAGINE[chiave]
    costo_totale = prezzo_unitario * n_immagini
    return {
        "modello": modello, "size": size, "quality": quality,
        "n_immagini": n_immagini, "prezzo_unitario_usd": prezzo_unitario,
        "costo_totale_usd": round(costo_totale, 4),
    }


# Riproduciamo l'esperimento prompt vago vs dettagliato della Sezione 3, ma via API
genera_immagine_api("Un robot", "robot_vago_api.png")
genera_immagine_api(
    "Un piccolo robot domestico dal design arrotondato e colori pastello, fermo su un "
    "pavimento di legno in un soggiorno luminoso, luce naturale da una finestra laterale, "
    "dettagli metallici riflettenti, stile illustrazione digitale morbida",
    "robot_dettagliato_api.png",
)

costo = calcola_costo_immagine("gpt-image-1", "1024x1024", "standard", n_immagini=2)
print(f"\nCosto stimato delle 2 immagini: ${costo['costo_totale_usd']:.4f}")


In [ ]:
def modifica_immagine_con_maschera_api(
    percorso_immagine: str, percorso_maschera_rgba: str, prompt: str, nome_file_output: str
) -> str:
    """Inpainting via API (Modulo 13): a differenza di diffusers, qui la maschera deve
    essere RGBA con canale alpha=0 (trasparente) nella zona da modificare, alpha=255 (opaco)
    altrove -- la convenzione OPPOSTA a quella binaria di diffusers vista in Sezione 5."""
    risultato = client.images.edit(
        model="gpt-image-1",
        image=open(percorso_immagine, "rb"),
        mask=open(percorso_maschera_rgba, "rb"),
        prompt=prompt,
        size="1024x1024",
    )
    immagine_bytes = base64.b64decode(risultato.data[0].b64_json)
    with open(nome_file_output, "wb") as f:
        f.write(immagine_bytes)
    print(f"Immagine modificata salvata in: {nome_file_output}")
    return nome_file_output


def crea_maschera_rgba_da_overlay(percorso_overlay: str, soglia: int = 15) -> str:
    """Converte lo stesso overlay usato in Sezione 5 nel formato RGBA richiesto dall'API
    OpenAI (alpha=0 nella zona da modificare, invece del bianco/nero di diffusers)."""
    overlay = Image.open(percorso_overlay).convert("RGB")
    arr = np.array(overlay)
    e_zona_da_modificare = np.all(arr < soglia, axis=-1)
    alpha = np.where(e_zona_da_modificare, 0, 255).astype(np.uint8)
    rgba = np.dstack([arr, alpha])
    percorso_output = "mesa_maschera_rgba_api.png"
    Image.fromarray(rgba, mode="RGBA").save(percorso_output)
    return percorso_output


maschera_rgba = crea_maschera_rgba_da_overlay(PERCORSO_OVERLAY)
modifica_immagine_con_maschera_api(
    percorso_immagine=PERCORSO_IMMAGINE,
    percorso_maschera_rgba=maschera_rgba,
    prompt=(
        "Un tavolo da pranzo in legno chiaro con sedie intorno, con al centro un grande "
        "vaso di ceramica bianca pieno di girasoli freschi, luce naturale, stile fotografico"
    ),
    nome_file_output="mesa_con_girasoli_api.png",
)


---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 23 — Progetto 2 — Modulo 23 del Corso Completo di IA Generativa*